# Mimicry Training on Colab

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

In [ ]:
from pathlib import Path

REPO_URL = "YOUR_REPO_URL"
PROJECT_DIR = Path('/content/Mimicry')
DRIVE_ROOT = Path('/content/drive/MyDrive/PyWatermark')

TRAIN_DIR = DRIVE_ROOT / 'datasets' / 'train'
VAL_DIR = DRIVE_ROOT / 'datasets' / 'val'
TEST_DIR = DRIVE_ROOT / 'datasets' / 'test'

CHECKPOINT_CLEAN = DRIVE_ROOT / 'artifacts' / 'checkpoints_clean'
LOG_CLEAN = DRIVE_ROOT / 'artifacts' / 'logs_clean'
CHECKPOINT_ROBUST = DRIVE_ROOT / 'artifacts' / 'checkpoints_robust_16'
LOG_ROBUST = DRIVE_ROOT / 'artifacts' / 'logs_robust_16'
PLOTS_ROBUST = DRIVE_ROOT / 'artifacts' / 'plots_robust_16'


## 3. Clone or update the repo

In [ ]:
if PROJECT_DIR.exists():
    %cd {PROJECT_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} Mimicry
    %cd {PROJECT_DIR}

!pip install -q -r requirements.txt

## 4. Prepare dataset

In [ ]:
%cd {PROJECT_DIR}
!python -m data.prepare_coco \
    --download-split val2017 \
    --raw-dir {DRIVE_ROOT / 'raw'} \
    --output-root {DRIVE_ROOT / 'datasets'} \
    --train-count 4000 \
    --val-count 500 \
    --test-count 500 \
    --force

## 5. Clean bootstrap

In [ ]:
%cd {PROJECT_DIR}
!python -m training.train \
    --train-dir {TRAIN_DIR} \
    --val-dir {VAL_DIR} \
    --test-dir {TEST_DIR} \
    --checkpoint-dir {CHECKPOINT_CLEAN} \
    --log-dir {LOG_CLEAN} \
    --epochs 10 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.10 \
    --invisibility-weight 0.25 \
    --detection-weight 8.0 \
    --disable-attacks

## 6. Robust training

In [ ]:
%cd {PROJECT_DIR}
!python -m training.train \
    --train-dir {TRAIN_DIR} \
    --val-dir {VAL_DIR} \
    --test-dir {TEST_DIR} \
    --checkpoint-dir {CHECKPOINT_ROBUST} \
    --log-dir {LOG_ROBUST} \
    --resume {CHECKPOINT_CLEAN / 'best.pt'} \
    --epochs 60 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.15 \
    --invisibility-weight 0.10 \
    --detection-weight 12.0 \
    --disable-amp

## 7. Resume robust training

In [ ]:
%cd {PROJECT_DIR}
!python -m training.train \
    --train-dir {TRAIN_DIR} \
    --val-dir {VAL_DIR} \
    --test-dir {TEST_DIR} \
    --checkpoint-dir {CHECKPOINT_ROBUST} \
    --log-dir {LOG_ROBUST} \
    --auto-resume \
    --epochs 60 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --key-bits 16 \
    --encoder-alpha 0.15 \
    --invisibility-weight 0.10 \
    --detection-weight 12.0 \
    --disable-amp

## 8. Export plots

In [ ]:
%cd {PROJECT_DIR}
!python -m evaluation.plot_training_curves \
    --history {LOG_ROBUST / 'metrics_history.csv'} \
    --output-dir {PLOTS_ROBUST} \
    --title-prefix "Mimicry robust_16"